# Ed-Fi Documentation Sitemap Analysis

This notebook extracts and analyzes all documentation URLs from the Ed-Fi documentation site using the sitemap.xml file.

In [2]:
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import os

In [3]:
def get_sitemap_urls(source_path_or_url):
    """
    Parses a Docusaurus sitemap.xml from a local path or a live URL.
    """
    if source_path_or_url.startswith('http'):
        response = requests.get(source_path_or_url)
        root = ET.fromstring(response.content)
    else:
        tree = ET.parse(source_path_or_url)
        root = tree.getroot()

    # Sitemaps use namespaces
    namespace = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
    
    urls = []
    for url_tag in root.findall('ns:url', namespace):
        loc = url_tag.find('ns:loc', namespace).text
        urls.append(loc)
    
    return urls

In [4]:
# Usage for your local repo
sitemap_path = r'c:\Dev\local-ed-fi\ed-fi-alliance-oss.github.io\build\sitemap.xml'

# Fallback to live site if local build is missing
if not os.path.exists(sitemap_path):
    print("Local sitemap not found. Using live site...")
    sitemap_path = 'https://docs.ed-fi.org/sitemap.xml'
else:
    print(f"Using local sitemap: {sitemap_path}")

doc_urls = get_sitemap_urls(sitemap_path)
print(f"Found {len(doc_urls)} URLs")

Using local sitemap: c:\Dev\local-ed-fi\ed-fi-alliance-oss.github.io\build\sitemap.xml
Found 1718 URLs


In [5]:
# Convert to DataFrame for analysis
df = pd.DataFrame(doc_urls, columns=['url'])

# Simple filtering to focus on specific doc sections
df['section'] = df['url'].apply(lambda x: x.split('/')[3] if len(x.split('/')) > 3 else 'root')

# Define sections to exclude (modify as needed)
excluded_sections = ['blog']

# Filter dataframe
df_filtered = df[~df['section'].isin(excluded_sections)].copy()

print(f"\nTotal pages found: {len(df)}")
print(f"Pages after filtering (excluded {excluded_sections}): {len(df_filtered)}")
print("\nPages by section (before filtering):")
print(df['section'].value_counts())
print("\nPages by section (after filtering):")
print(df_filtered['section'].value_counts())


Total pages found: 1718
Pages after filtering (excluded ['blog']): 1659

Pages by section (before filtering):
section
reference          1223
getting-started     257
partners            148
blog                 59
community            29
search                1
                      1
Name: count, dtype: int64

Pages by section (after filtering):
section
reference          1223
getting-started     257
partners            148
community            29
search                1
                      1
Name: count, dtype: int64


In [6]:
# Display first 10 URLs from filtered data
df_filtered.head(10)

,url,section
59,https://docs.ed-fi.org/community,community
60,https://docs.ed-fi.org/getting-started,getting-started
61,https://docs.ed-fi.org/partners,partners
62,https://docs.ed-fi.org/reference,reference
63,https://docs.ed-fi.org/search,search
64,https://docs.ed-fi.org/community/involved/,community
65,https://docs.ed-fi.org/community/involved/ceds,community
66,https://docs.ed-fi.org/community/involved/code...,community
67,https://docs.ed-fi.org/community/involved/comm...,community
68,https://docs.ed-fi.org/community/involved/dict...,community


In [ ]:
# Save filtered data to CSV
import os
os.makedirs('data/processed', exist_ok=True)
df_filtered.to_csv('data/processed/edfi_doc_urls.csv', index=False)
print(f"Saved {len(df_filtered)} URLs to data/processed/edfi_doc_urls.csv")